# Factor evaluation

Display 50 completed residual-augmentation comparisons (24 in A, 26 in B) from the three TEST summary tables. Factor blocks augment fixed GlucoBench predictions; L/R results are validation-only. The default path reads tables without fitting or resampling.

The 42 manual ablations fit glucose and manual features jointly with Ridge penalties {0.1, 1, 10}. The 50 factor comparisons add residual blocks to a fixed GlucoBench baseline, using 13 penalties from 0.001 to 1,000. Input representations, selection and some subject rules differ; the historical Protocol A RMSE gap of approximately 0.382 mg/dL is descriptive and cannot be attributed to residual modeling alone.

The 20-window rule fixes eligible validation subjects before factor selection (`selection_rows`); common-core refits use all core validation subjects before reporting filters (`core_rows`). Evaluation also applies the rule within each reported subset. Manual ablation did not use this selection cutoff; all subjects in both saved core TEST comparisons meet it.


## Inputs and configurations

The default reads saved summaries and the completion record. Explicit validation reconstruction uses TRAIN/VALIDATION inputs, stored OOF residuals and fixed penalties; choose `REBUILD_VALIDATION=True` and a new directory under `reproductions/`.


In [1]:
# --- Imports, paths and optional validation reconstruction ---
import json
from pathlib import Path

import pandas as pd
from IPython.display import display
from src.paths import PROJECT_ROOT


FACTOR_DIR = PROJECT_ROOT / "results" / "factor_discovery"
RESULT_DIR = PROJECT_ROOT / "results" / "factor_evaluation"
REBUILD_VALIDATION = False
RECONSTRUCTION_DIR = None  # set a new path under PROJECT_ROOT / "reproductions" before selecting reconstruction
pd.set_option("display.max_columns", 16)
pd.set_option("display.width", 160)

In [2]:
# --- Comparison definitions and completed results ---
evaluation_plan = json.loads((FACTOR_DIR / "evaluation_plan.json").read_text(encoding="utf-8"))
completion = json.loads((RESULT_DIR / "completed_evaluation.json").read_text(encoding="utf-8"))
gains = pd.read_csv(RESULT_DIR / "comparison_results.csv")
metrics = pd.read_csv(RESULT_DIR / "configuration_metrics.csv")
meal_strata = pd.read_csv(RESULT_DIR / "meal_strata.csv")
from src.comparison_registry import validate_saved_results
comparisons = validate_saved_results(evaluation_plan, completion, gains, metrics, meal_strata)
display(comparisons.groupby(["family", "protocol_tag"]).size().unstack(fill_value=0))

protocol_tag,A,B
family,,
E1,7,7
E5,2,4
E8 minimal,4,4
E9,4,4
primary,4,4
secondary E2,3,3


## TEST comparison results

Comparisons condition on fixed predictions from TRAIN-only fits, without TRAIN+VALIDATION refitting. TEST was used earlier; these are descriptive comparisons, not independent confirmation.

\[
\Delta\mathrm{RMSE}=\mathrm{RMSE}_{\mathrm{to}}-\mathrm{RMSE}_{\mathrm{from}}.
\]

Negative values indicate lower error. Intervals use 2,000 paired subject resamples (seed 42). The applied rule requires at least 20 windows per subject. `p_bootstrap` is approximate; `p_holm_primary_family` adjusts P1-P4 within each protocol, with nominal error control unverified. The 0.1 mg/dL label is descriptive, not clinical or a significance threshold.


In [3]:
# --- Primary contrasts on the core cohort ---
main = gains[gains["cohort"].eq("core") & gains["stratum"].eq("all") & gains["group"].eq("all") & gains["min_window_rule"].str.startswith("applied")]
primary = main[main["family"].eq("primary")]
display(primary[["id", "from_config", "to_config", "subjects", "rmse_from", "rmse_to", "d_rmse", "d_rmse_low", "d_rmse_high", "p_holm_primary_family"]].round(3))

,id,from_config,to_config,subjects,rmse_from,rmse_to,d_rmse,d_rmse_low,d_rmse_high,p_holm_primary_family
0,A:P1,base,base + M,44,10.957,10.910,-0.047,-0.170,0.084,0.462
10,A:P2,base,base + T,44,10.957,10.895,-0.063,-0.120,-0.010,0.072
20,A:P3,base + M,base + M + T,44,10.910,10.877,-0.033,-0.073,0.013,0.318
30,A:P4,base,base + O,44,10.957,10.822,-0.135,-0.229,-0.042,0.020
120,B:P1,base,base + M,9,11.327,11.081,-0.245,-0.362,-0.137,0.004
130,B:P2,base,base + T,9,11.327,11.252,-0.075,-0.116,-0.021,0.030
140,B:P3,base + M,base + M + T,9,11.081,11.017,-0.064,-0.114,-0.007,0.033
150,B:P4,base,base + O,9,11.327,11.082,-0.244,-0.349,-0.145,0.004


In [4]:
# --- Saved configuration and meal-stratum summaries ---
display(metrics[metrics["cohort"].eq("core") & metrics["config"].isin(["base", "base + M", "base + T", "base + O"])][["protocol_tag", "config", "subjects_ge_20", "windows", "rmse_subject_equal_rule_applied", "rmse_pooled"]].round(3))
display(meal_strata[meal_strata["family"].eq("primary") & meal_strata["cohort"].eq("core") & meal_strata["group"].eq("all") & meal_strata["min_window_rule"].str.startswith("applied")][["id", "stratum", "subjects", "d_rmse", "d_rmse_low", "d_rmse_high"]].round(3))

,protocol_tag,config,subjects_ge_20,windows,rmse_subject_equal_rule_applied,rmse_pooled
12,A,base,44,21083,10.957,11.065
13,A,base + M,44,21083,10.910,10.991
19,A,base + O,44,21083,10.822,10.927
20,A,base + T,44,21083,10.895,11.010
66,B,base,9,26861,11.327,11.380
67,B,base + M,9,26861,11.081,11.149
73,B,base + O,9,26861,11.082,11.141
74,B,base + T,9,26861,11.252,11.311


,id,stratum,subjects,d_rmse,d_rmse_low,d_rmse_high
0,A:P1,P30,25,-0.199,-1.272,0.977
2,A:P1,P31_120,34,-0.006,-0.120,0.121
4,A:P1,NoMeal120,44,-0.007,-0.093,0.103
6,A:P2,P30,25,-0.131,-0.456,0.211
8,A:P2,P31_120,34,0.056,-0.082,0.194
10,A:P2,NoMeal120,44,-0.065,-0.125,-0.008
12,A:P3,P30,25,0.171,-0.090,0.550
14,A:P3,P31_120,34,-0.092,-0.210,0.029
16,A:P3,NoMeal120,44,-0.042,-0.073,-0.010
18,A:P4,P30,25,-0.506,-1.367,0.422


## Optional validation reconstruction

Rebuild factors from the selected recipes and compare window keys, values and missingness with independent materialized inputs. Check validation references and meal strata before writing CSVs. E9 retains its specified meal-delay semantics.

Mixed-source filtering provides statistical isolation, not physical isolation of decoded rows.


In [5]:
# --- Assemble admitted train/validation windows with model configurations ---
if REBUILD_VALIDATION:
    from src import factor_evaluation as FE, factor_validation as FG, window_data as W
    if RECONSTRUCTION_DIR is None:
        raise ValueError("choose a new RECONSTRUCTION_DIR under reproductions/")
    out = Path(RECONSTRUCTION_DIR).resolve()
    if not out.is_relative_to((PROJECT_ROOT / "reproductions").resolve()) or out.exists():
        raise ValueError("reconstruction needs a new directory under reproductions/")
    out.mkdir(parents=True)
    assembled = {tag: FE.assemble_protocol(tag, evaluation_plan, ("train", "validation"), W.Access()) for tag in ("A", "B")}
    for tag, asm in assembled.items():
        checks = FG.data_range_check(tag, asm, PROJECT_ROOT)
        if not all(value for value in checks.values() if isinstance(value, bool)):
            raise ValueError(f"input range checks failed: {checks}")

In [6]:
# --- Rebuilt factors against saved values, missingness and window keys  ---
if REBUILD_VALIDATION:
    compare_factor_values = pd.concat([FG.compare_factor_values(tag, asm, evaluation_plan, FACTOR_DIR, (asm["keys"], asm["minutes"], asm["events"])) for tag, asm in assembled.items()], ignore_index=True)
    if not compare_factor_values["passed"].all():
        raise ValueError("rebuilt factors differ in values, missingness or window keys")

In [7]:
# --- Validation contrasts; the engine caches each configuration/cohort fit ---
if REBUILD_VALIDATION:
    results = {tag: FE.evaluate_protocol(tag, evaluation_plan, asm, FE.N_RESAMPLES) for tag, asm in assembled.items()}
    validation = pd.concat([c.assign(protocol_tag=tag) for tag, (c, _) in results.items()], ignore_index=True)
    summary = pd.concat([s.assign(protocol_tag=tag) for tag, (_, s) in results.items()], ignore_index=True)


In [8]:
# --- Saved validation references, primary family and meal strata ---
if REBUILD_VALIDATION:
    references = {}
    compare_validation_results = pd.concat([FG.compare_validation_results(c, evaluation_plan, FACTOR_DIR, references) for c, _ in results.values()], ignore_index=True)
    if not compare_validation_results["status"].isin(["match", "match (point estimate)"]).all():
        raise ValueError("validation results differ from their saved references")
    for tag, asm in assembled.items():
        masks = FE.strata_masks(asm["table"])
        if not (sum(mask.astype(int) for mask in masks.values()) == 1).all():
            raise ValueError(f"meal strata do not partition protocol {tag}")
    validation.assign(status="VALIDATION reconstruction; not a new TEST result").to_csv(out / "validation_contrasts.csv", index=False)
    summary.to_csv(out / "validation_summary.csv", index=False)
    display(validation[validation["cohort"].eq("core") & validation["stratum"].eq("all") & validation["group"].eq("all") & validation["min_window_rule"].str.startswith("applied")][["id", "d_rmse", "d_rmse_low", "d_rmse_high"]].round(3))